In [35]:
import numpy as np
import pandas as pd
import torch
from torch import nn

data = pd.read_csv('train.csv')

In [36]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [37]:
data = data.to_numpy()
# 0th col are the labels of the images
# 1st col up to 784th col are pixel values

In [38]:
data

array([[1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [7, 0, 0, ..., 0, 0, 0],
       [6, 0, 0, ..., 0, 0, 0],
       [9, 0, 0, ..., 0, 0, 0]], shape=(42000, 785))

In [39]:
labels = data[:, 0]
data = data[:, 1:785]

In [40]:
data.shape

(42000, 784)

In [41]:
train_split = int(0.8 * len(data))
y_train, y_test = labels[:train_split], labels[train_split:]
X_train, X_test = data[:train_split], data[train_split:]

In [42]:
X_train, X_test = torch.tensor(X_train, dtype=torch.float32), torch.tensor(X_test, dtype=torch.float32)
y_train, y_test = torch.tensor(y_train, dtype=torch.int64), torch.tensor(y_test, dtype=torch.int64)
X_train, X_test = X_train/255, X_test/255

In [43]:
class NumClassificationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 10)
        self.layer2 = nn.Linear(10, 10)
        self.layer3 = nn.Linear(10,10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.layer3(x)
        return x

In [44]:
model_1 = NumClassificationModel()

In [45]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model_1.parameters(), lr = 0.1)

In [46]:
X_train.shape

torch.Size([33600, 784])

In [47]:
y_logits = model_1(X_train)
print(y_logits, y_logits.shape)
y_pred = nn.Softmax(dim=1)(y_logits)
print(y_pred, y_pred.shape)
y_pred_labels = torch.argmax(y_pred, dim=1)
print(y_pred_labels, y_pred_labels.shape)

tensor([[-0.2592,  0.0232,  0.2449,  ...,  0.2437,  0.1363, -0.1841],
        [-0.2586,  0.0177,  0.2431,  ...,  0.2459,  0.1282, -0.1888],
        [-0.2652,  0.0130,  0.2448,  ...,  0.2495,  0.1319, -0.1904],
        ...,
        [-0.2489,  0.0216,  0.2455,  ...,  0.2469,  0.1205, -0.1905],
        [-0.2703,  0.0164,  0.2481,  ...,  0.2493,  0.1439, -0.1857],
        [-0.2512,  0.0303,  0.2432,  ...,  0.2387,  0.1338, -0.1811]],
       grad_fn=<AddmmBackward0>) torch.Size([33600, 10])
tensor([[0.0783, 0.1038, 0.1296,  ..., 0.1295, 0.1163, 0.0844],
        [0.0785, 0.1035, 0.1297,  ..., 0.1301, 0.1156, 0.0842],
        [0.0781, 0.1031, 0.1301,  ..., 0.1307, 0.1162, 0.0842],
        ...,
        [0.0792, 0.1038, 0.1299,  ..., 0.1301, 0.1146, 0.0840],
        [0.0775, 0.1033, 0.1302,  ..., 0.1303, 0.1173, 0.0844],
        [0.0788, 0.1044, 0.1292,  ..., 0.1286, 0.1158, 0.0845]],
       grad_fn=<SoftmaxBackward0>) torch.Size([33600, 10])
tensor([2, 7, 7,  ..., 7, 7, 2]) torch.Size([33600])

In [48]:
def accuracy(actual_labels, pred_labels):
    '''
    both actual_labels and pred_labels are vectors where each entry is an example
    '''
    return (pred_labels == actual_labels).sum().item() / len(actual_labels) * 100

In [54]:
# training loop
epochs = 1000 

for epoch in range(epochs):
    model_1.train()
    
    # forward pass
    y_logits = model_1(X_train)
    y_pred = nn.Softmax(dim=1)(y_logits)
    y_pred_labels = torch.argmax(y_pred, dim=1)

    # calculate the loss
    loss = loss_fn(y_logits, y_train)
    acc = accuracy(y_train, y_pred_labels)

    # optmizier.zero_grad()
    optimizer.zero_grad()

    # loss.backward()
    loss.backward()

    # optimizer.step()
    optimizer.step()

    # testing loop
    model_1.eval()
    with torch.inference_mode():
        # forward pass
        y_test_logits = model_1(X_test)
        y_test_pred = nn.Softmax(dim=1)(y_test_logits)
        y_test_pred_labels = torch.argmax(y_test_pred, dim=1)

        # calculate the loss
        test_loss = loss_fn(y_test_logits, y_test)
        test_acc = accuracy(y_test, y_test_pred_labels)

        if epoch % 20 == 0: 
            print(f'epoch: {epoch} | train loss: {loss:.5f} | train acc: {acc}% | test loss: {test_loss:.5f} | test acc: {test_acc}%')


epoch: 0 | train loss: 0.16528 | train acc: 95.02083333333333% | test loss: 0.23040 | test acc: 93.75%
epoch: 20 | train loss: 0.16618 | train acc: 94.9672619047619% | test loss: 0.23134 | test acc: 93.69047619047619%
epoch: 40 | train loss: 0.16342 | train acc: 95.10416666666667% | test loss: 0.22883 | test acc: 93.85714285714286%
epoch: 60 | train loss: 0.16145 | train acc: 95.19345238095238% | test loss: 0.22714 | test acc: 93.82142857142857%
epoch: 80 | train loss: 0.16068 | train acc: 95.19345238095238% | test loss: 0.22661 | test acc: 93.84523809523809%
epoch: 100 | train loss: 0.16035 | train acc: 95.22321428571429% | test loss: 0.22650 | test acc: 93.80952380952381%
epoch: 120 | train loss: 0.16031 | train acc: 95.21130952380953% | test loss: 0.22672 | test acc: 93.80952380952381%
epoch: 140 | train loss: 0.16089 | train acc: 95.19047619047619% | test loss: 0.22758 | test acc: 93.78571428571428%
epoch: 160 | train loss: 0.16273 | train acc: 95.125% | test loss: 0.22966 | test a